# 🔮 Inference and Summary Report for Stance Classification
This notebook loads the best models from each fold and:
- Aggregates predictions
- Plots a confusion matrix and metrics per fold
- Generates a summary report
- Exports all results to CSV and PDF

In [ ]:
# 📦 Install needed packages
!pip install pandas scikit-learn matplotlib seaborn fpdf -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from fpdf import FPDF
import os

In [ ]:
# 📊 Load predictions from each fold
all_predictions = []
results = []
for fold in range(1, 6):
    pred_path = f"output/fold{fold}/predictions.csv"
    df = pd.read_csv(pred_path)
    df['fold'] = fold
    all_predictions.append(df)

    y_true = df['true_label']
    y_pred = df['predicted_label']
    acc = (y_true == y_pred).mean()
    f1 = f1_score(y_true, y_pred, average='weighted')
    results.append({"fold": fold, "accuracy": acc, "f1": f1})

predictions_df = pd.concat(all_predictions)
metrics_df = pd.DataFrame(results)
predictions_df.to_csv("output/all_folds_predictions.csv", index=False)
metrics_df.to_csv("output/summary_metrics.csv", index=False)

In [ ]:
# 🔍 Plot Confusion Matrix of last fold
y_true = df['true_label']
y_pred = df['predicted_label']
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Negative", "Positive", "Neutral"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix - Fold 5")
plt.savefig("output/confusion_matrix_fold5.png")
plt.show()

In [ ]:
# 📄 Generate PDF report
pdf = FPDF()
pdf.add_page()
pdf.set_font("Arial", 'B', 16)
pdf.cell(0, 10, "Stance Classification Report", ln=True, align='C')
pdf.ln(10)

pdf.set_font("Arial", '', 12)
for _, row in metrics_df.iterrows():
    pdf.cell(0, 10, f"Fold {int(row['fold'])} - Accuracy: {row['accuracy']:.3f} - F1: {row['f1']:.3f}", ln=True)

pdf.ln(10)
pdf.image("output/confusion_matrix_fold5.png", w=150)

pdf.output("output/stance_classification_summary.pdf")